In [1]:
import torch
import numpy as np
import random
import json
import time

from pathlib import Path

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

with open(DATA_DIR / "vocab.json", encoding="utf-8") as f:
    word_to_id = json.load(f)

token_data = torch.load(
    DATA_DIR / "token_ids.pt",
    weights_only=True
)

train_ids = token_data["train"]

VOCAB_SIZE = len(word_to_id)
EMBEDDING_DIM = 100

print("Vocabulary size:", VOCAB_SIZE)
print("Training tokens:", len(train_ids))
print("Embedding dimension:", EMBEDDING_DIM)

Vocabulary size: 10000
Training tokens: 2051910
Embedding dimension: 100


In [2]:
from gensim.models import Word2Vec

id_to_word = {
    idx: word for word, idx in word_to_id.items()
}

# Split the training corpus into chunks
CHUNK_SIZE = 100

sentences = []

for start in range(0, len(train_ids), CHUNK_SIZE):
    chunk = train_ids[start:start + CHUNK_SIZE]

    words = [
        id_to_word[token_id]
        for token_id in chunk
        if token_id != word_to_id["<pad>"]
    ]

    if len(words) >= 2:
        sentences.append(words)

print("Number of training sequences:", len(sentences))
print("First 20 words:")
print(sentences[0][:20])

Number of training sequences: 20520
First 20 words:
['=', 'valkyria', 'chronicles', 'iii', '=', '<unk>', 'no', 'valkyria', '3', ':', '<unk>', 'chronicles', '(', 'japanese', ':', '<unk>', ',', 'lit', '.', 'valkyria']


In [3]:
from gensim.models import Word2Vec
import time

start_time = time.perf_counter()

skipgram_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    negative=5,
    hs=0,
    epochs=5,
    workers=1,
    seed=42
)

skipgram_training_time = time.perf_counter() - start_time

print("Skip-gram training completed!")
print(f"Training time: {skipgram_training_time:.2f} seconds")
print(f"Training time: {skipgram_training_time / 60:.2f} minutes")
print("Embedding dimension:", skipgram_model.vector_size)
print("Words learned:", len(skipgram_model.wv))


Skip-gram training completed!
Training time: 41.76 seconds
Training time: 0.70 minutes
Embedding dimension: 100
Words learned: 9999


In [4]:
import numpy as np

word = "game"

vector = skipgram_model.wv[word]

print("Word:", word)
print("Embedding shape:", vector.shape)
print("First 10 values:")
print(np.round(vector[:10], 4))

print("\nMost similar words:")

for similar_word, similarity in skipgram_model.wv.most_similar(
    word, topn=10
):
    print(f"{similar_word:20s} {similarity:.4f}")

Word: game
Embedding shape: (100,)
First 10 values:
[-0.1583 -0.0535  0.0669  0.3318 -0.1516  0.0788 -0.2445  0.1753 -0.841
  0.5418]

Most similar words:
games                0.7459
franchise            0.6697
gameplay             0.6631
multiplayer          0.6441
3d                   0.6400
console              0.6307
playstation          0.6261
player               0.6247
crush                0.6243
psp                  0.6194


In [5]:
import numpy as np
import torch

PAD_ID = word_to_id["<pad>"]

# Initialize embedding matrix
embedding_matrix = np.zeros(
    (VOCAB_SIZE, EMBEDDING_DIM),
    dtype=np.float32
)

missing_words = []

for word, idx in word_to_id.items():

    if word == "<pad>":
        continue

    if word in skipgram_model.wv:
        embedding_matrix[idx] = skipgram_model.wv[word]
    else:
        missing_words.append(word)

print("Embedding matrix shape:", embedding_matrix.shape)
print("Missing words:", len(missing_words))
print("Missing word examples:", missing_words[:10])

print(
    "Padding vector is zero:",
    np.all(embedding_matrix[PAD_ID] == 0)
)

Embedding matrix shape: (10000, 100)
Missing words: 0
Missing word examples: []
Padding vector is zero: True


In [6]:
game_id = word_to_id["game"]

matrix_vector = embedding_matrix[game_id]
skipgram_vector = skipgram_model.wv["game"]

assert np.allclose(
    matrix_vector,
    skipgram_vector
)

print("Game token ID:", game_id)
print("Embedding shape:", matrix_vector.shape)
print("Vocabulary alignment verified!")

Game token ID: 71
Embedding shape: (100,)
Vocabulary alignment verified!


In [7]:
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Save embedding matrix
torch.save(
    torch.from_numpy(embedding_matrix),
    MODELS_DIR / "skipgram_embeddings.pt"
)

# Save original gensim model
skipgram_model.save(
    str(MODELS_DIR / "skipgram_word2vec.model")
)

# Save embedding training metadata
embedding_metadata = {
    "method": "Skip-gram with Negative Sampling",
    "dataset": "WikiText-2 training split",
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "window": 5,
    "negative_samples": 5,
    "epochs": 5,
    "seed": 42,
    "training_time_seconds": skipgram_training_time,
    "words_learned": len(skipgram_model.wv),
    "missing_non_padding_words": len(missing_words)
}

with open(
    RESULTS_DIR / "skipgram_results.json",
    "w"
) as f:
    json.dump(embedding_metadata, f, indent=2)

print("Skip-gram embeddings saved successfully!")
print("Embedding matrix:", MODELS_DIR / "skipgram_embeddings.pt")

Skip-gram embeddings saved successfully!
Embedding matrix: c:\Users\User\NUS\DSA NLP\DSA4213_Assignment1\models\skipgram_embeddings.pt
